# Notebook 03 — MongoDB Storage
**Unit III: NoSQL Databases and Big Data Storage**

Covers:
- Why NoSQL for Big Data?
- Inserting `ipl.csv` into MongoDB using `pymongo`
- MapReduce-style aggregation using MongoDB Aggregation Pipeline
- Querying and retrieving results

## 3.0 Why MongoDB for IPL Data?

| Feature | MongoDB | Traditional RDBMS |
|---------|---------|------------------|
| Schema | Flexible (JSON-like documents) | Fixed schema |
| Scale | Horizontal sharding | Vertical scaling |
| Speed | Fast reads on large collections | Slower with joins |
| Format | BSON (Binary JSON) | Tables/rows |

> MongoDB is a **NoSQL** database ideal for unstructured/semi-structured Big Data. Each IPL delivery record becomes a **document** in a **collection**.

### Setup
```bash
# Install pymongo
pip install pymongo

# Start MongoDB locally (if using Docker)
docker run -d -p 27017:27017 --name mongodb mongo

# OR use MongoDB Atlas (free cloud tier): https://www.mongodb.com/atlas
```

In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import ConnectionFailure, BulkWriteError
import json
import warnings
warnings.filterwarnings('ignore')

print('pymongo imported successfully.')

pymongo imported successfully.


In [2]:
# ── Connect to MongoDB ────────────────────────────────────────────────────────
# Local MongoDB
MONGO_URI = 'mongodb://localhost:27017/'

# If using Atlas, replace with your connection string:
# MONGO_URI = 'mongodb+srv://<user>:<password>@cluster0.xxxxx.mongodb.net/'

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')  # Check connection
    print('Connected to MongoDB successfully!')
except ConnectionFailure as e:
    print(f'Connection failed: {e}')
    print('Make sure MongoDB is running (docker run -d -p 27017:27017 --name mongodb mongo)')

Connection failed: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 69e7cdc04ea3b916edb164aa, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
Make sure MongoDB is running (docker run -d -p 27017:27017 --name mongodb mongo)


In [3]:
# ── Create Database and Collection ───────────────────────────────────────────
db = client['ipl_bigdata']
collection = db['deliveries']

print(f'Database  : {db.name}')
print(f'Collection: {collection.name}')

Database  : ipl_bigdata
Collection: deliveries


## 3.1 Insert IPL Data into MongoDB

In [4]:
# Load cleaned CSV
df = pd.read_csv('../data/ipl_clean.csv')
print(f'Loaded {len(df):,} records from ipl_clean.csv')

# Convert NaN to None (MongoDB compatible)
df = df.where(pd.notnull(df), None)

# Convert to list of dicts (documents)
records = df.to_dict(orient='records')
print(f'Sample document:\n{json.dumps(records[0], indent=2, default=str)}')

Loaded 283,678 records from ipl_clean.csv
Sample document:
{
  "Unnamed: 0": 141607,
  "match_id": 335982,
  "date": "2008-04-18",
  "match_type": "T20",
  "event_name": "Indian Premier League",
  "innings": 1,
  "batting_team": "Kolkata Knight Riders",
  "bowling_team": "Royal Challengers Bangalore",
  "over": 0,
  "ball": 1,
  "ball_no": 0.1,
  "batter": "SC Ganguly",
  "bat_pos": 1,
  "runs_batter": 0,
  "balls_faced": 1,
  "bowler": "P Kumar",
  "valid_ball": 1,
  "runs_extras": 1,
  "runs_total": 1,
  "runs_bowler": 0,
  "runs_not_boundary": false,
  "extra_type": "legbyes",
  "non_striker": "BB McCullum",
  "non_striker_pos": 2,
  "wicket_kind": "Unknown",
  "player_out": "Unknown",
  "fielders": "Unknown",
  "runs_target": 0.0,
  "review_batter": "Unknown",
  "team_reviewed": "Unknown",
  "review_decision": "Unknown",
  "umpire": "Unknown",
  "umpires_call": false,
  "player_of_match": "BB McCullum",
  "match_won_by": "Kolkata Knight Riders",
  "win_outcome": "140 runs",
  "toss

In [5]:
# ── Drop existing collection and insert fresh ─────────────────────────────────
collection.drop()
print('Existing collection dropped.')

# Insert in batches of 10,000 for performance
BATCH_SIZE = 10_000
total_inserted = 0

for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i + BATCH_SIZE]
    result = collection.insert_many(batch)
    total_inserted += len(result.inserted_ids)
    print(f'  Inserted batch {i//BATCH_SIZE + 1}: {total_inserted:,} records so far...')

print(f'\nTotal documents inserted: {total_inserted:,}')
print(f'Collection count: {collection.count_documents({}):,}')

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 69e7cdc04ea3b916edb164aa, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

In [ ]:
# ── Create indexes for faster queries ────────────────────────────────────────
# Auto-detect key columns
cols = df.columns.tolist()
index_cols = []

for possible in ['match_id', 'season', 'batter', 'batsman', 'bowler', 'batting_team']:
    if possible in cols:
        collection.create_index(possible)
        index_cols.append(possible)

print(f'Indexes created on: {index_cols}')

## 3.2 Basic CRUD Queries

In [ ]:
# READ — Fetch one document
doc = collection.find_one()
print('One document from MongoDB:')
print(json.dumps({k: str(v) for k, v in doc.items() if k != '_id'}, indent=2))

In [ ]:
# READ — Filter: all deliveries by a team (update column name as needed)
team_col = 'batting_team' if 'batting_team' in df.columns else 'team1'

if team_col in df.columns:
    sample_team = df[team_col].dropna().unique()[0]
    count = collection.count_documents({team_col: sample_team})
    print(f'Documents for "{sample_team}": {count:,}')

## 3.3 MapReduce-Style Aggregation Pipeline

MongoDB's **Aggregation Pipeline** is the modern equivalent of Hadoop MapReduce:

| MapReduce Phase | MongoDB Equivalent |
|----------------|-------------------|
| Map | `$group` (key extraction) |
| Reduce | `$sum`, `$avg`, `$max` |
| Filter | `$match` |
| Sort | `$sort` |
| Limit | `$limit` |

In [ ]:
# ── Aggregation 1: Top 10 run scorers ────────────────────────────────────────
batter_col = 'batter' if 'batter' in df.columns else 'batsman'
runs_col   = 'batter_runs' if 'batter_runs' in df.columns else 'batsman_runs'

if batter_col in df.columns:
    pipeline_scorers = [
        {'$match': {runs_col: {'$gt': 0}}},              # Map: filter non-zero runs
        {'$group': {
            '_id': f'${batter_col}',                     # Map: group by batter
            'total_runs': {'$sum': f'${runs_col}'},      # Reduce: sum runs
            'balls_faced': {'$sum': 1}                   # Reduce: count balls
        }},
        {'$addFields': {
            'strike_rate': {
                '$round': [{'$multiply': [{'$divide': ['$total_runs', '$balls_faced']}, 100]}, 2]
            }
        }},
        {'$sort': {'total_runs': -1}},                   # Sort descending
        {'$limit': 10}                                   # Top 10 only
    ]

    top_scorers_mongo = list(collection.aggregate(pipeline_scorers))
    result_df = pd.DataFrame(top_scorers_mongo).rename(columns={'_id': 'Batter'})
    print('Top 10 Run Scorers (via MongoDB Aggregation Pipeline):')
    print(result_df.to_string(index=False))
else:
    print('Batter column not found.')

In [ ]:
# ── Aggregation 2: Top 10 wicket takers ──────────────────────────────────────
wicket_col = 'is_wicket' if 'is_wicket' in df.columns else None
if 'player_dismissed' in df.columns and wicket_col is None:
    wicket_col = 'is_wicket'   # We created this in Notebook 01 cleaning step

if wicket_col and 'bowler' in df.columns:
    pipeline_wickets = [
        {'$match': {wicket_col: 1}},
        {'$group': {
            '_id': '$bowler',
            'wickets': {'$sum': 1},
            'balls_bowled': {'$sum': 1}
        }},
        {'$sort': {'wickets': -1}},
        {'$limit': 10}
    ]

    top_bowlers_mongo = list(collection.aggregate(pipeline_wickets))
    result_df2 = pd.DataFrame(top_bowlers_mongo).rename(columns={'_id': 'Bowler'})
    print('Top 10 Wicket Takers (via MongoDB Aggregation Pipeline):')
    print(result_df2.to_string(index=False))
else:
    print('Wicket or bowler column not found.')

In [ ]:
# ── Aggregation 3: Season-wise total runs ────────────────────────────────────
if 'season' in df.columns and runs_col in df.columns:
    pipeline_season = [
        {'$group': {
            '_id': '$season',
            'total_runs': {'$sum': f'${runs_col}'},
            'total_balls': {'$sum': 1}
        }},
        {'$sort': {'_id': 1}}
    ]

    season_data = list(collection.aggregate(pipeline_season))
    result_df3 = pd.DataFrame(season_data).rename(columns={'_id': 'Season'})
    print('Season-wise Run Totals (MongoDB Aggregation):')
    print(result_df3.to_string(index=False))
else:
    print('Season or runs column not found.')

In [ ]:
# ── Close connection ──────────────────────────────────────────────────────────
client.close()
print('MongoDB connection closed.')

## 3.4 Summary

| Step | Action |
|------|--------|
| Connect | `MongoClient` to local/Atlas MongoDB |
| Store | `insert_many()` in batches for performance |
| Index | Created indexes on key fields for fast queries |
| Aggregate | Used `$match → $group → $sort → $limit` pipeline |

> The aggregation pipeline mimics **MapReduce**: `$group` = Map, `$sum/$avg` = Reduce, `$match` = Filter.